In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col

In [0]:
# Define table names using environment variables
silver_drivers_table = f"{catalog_name}.{silver_schema}.drivers"
gold_ref_nationality_region_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"
gold_dim_drivers_table = f"{catalog_name}.{gold_schema}.dim_drivers"

# Read silver drivers table
drivers_df = spark.table(silver_drivers_table)

# Read gold ref_nationality_region table
ref_nationality_region_df = spark.table(gold_ref_nationality_region_table)

In [0]:
# Join drivers with ref_nationality_region on nationality
dim_drivers_df = (
    drivers_df.alias("driver")
    .join(
        ref_nationality_region_df.alias("ref_nationality_region"),
        col("driver.nationality") == col("ref_nationality_region.nationality"),
        "left"
    )
    # Select only required columns
    .select(
        col("driver.driver_id"),
        col("driver.driver_name"),
        col("driver.date_of_birth"),
        col("driver.nationality"),
        col("ref_nationality_region.region").alias("nationality_region")
    )
)

In [0]:
# Write transformed data to gold layer
(
    dim_drivers_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(gold_dim_drivers_table)
)

print(f"✓ Drivers dimension successfully written to {gold_dim_drivers_table}")

In [0]:
# Preview the drivers dimension table
#spark.table(gold_dim_drivers_table).limit(10).display()